In [1]:
import pandas as pd
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
import os


pogoda = pd.read_csv('DANE/WeatherNYC_2018.csv')
pogoda['time'] = pd.to_datetime(pogoda['time'])
pogoda['month'] = pogoda['time'].dt.month
pogoda['day'] = pogoda['time'].dt.day

srednia_pogoda = pogoda.groupby(['month', 'day'])[['temperature_2m (°C)', 'precipitation (mm)', 'wind_speed_10m (km/h)']].mean().reset_index()
srednia_pogoda.rename(columns={
    'temperature_2m (°C)': 'temperatura',
    'precipitation (mm)': 'opady',
    'wind_speed_10m (km/h)': 'wiatr'
}, inplace=True)

pliki = [
    'DANE/JC-202501-citibike-tripdata.csv',
    'DANE/JC-202504-citibike-tripdata.csv',
    'DANE/JC-202507-citibike-tripdata.csv',
    'DANE/JC-202510-citibike-tripdata.csv'
]

lista_df = [pd.read_csv(plik) for plik in pliki]
rowery = pd.concat(lista_df, ignore_index=True)

rowery['started_at'] = pd.to_datetime(rowery['started_at'])
rowery['month'] = rowery['started_at'].dt.month
rowery['day'] = rowery['started_at'].dt.day

trasy_dzienne = rowery.groupby(['month', 'day', 'start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
polaczone = pd.merge(trasy_dzienne, srednia_pogoda, on=['month', 'day'])

najwyzsza_temp = polaczone.sort_values(by=['temperatura', 'liczba_tras'], ascending=[False, False]).head(100)
najnizsza_temp = polaczone.sort_values(by=['temperatura', 'liczba_tras'], ascending=[True, False]).head(100)
najwieksze_opady = polaczone.sort_values(by=['opady', 'liczba_tras'], ascending=[False, False]).head(100)
najmniejsze_opady = polaczone.sort_values(by=['opady', 'liczba_tras'], ascending=[True, False]).head(100)
najsilniejszy_wiatr = polaczone.sort_values(by=['wiatr', 'liczba_tras'], ascending=[False, False]).head(100)
najslabszy_wiatr = polaczone.sort_values(by=['wiatr', 'liczba_tras'], ascending=[True, False]).head(100)

najwyzsza_temp.to_csv('WYNIKI/POMOCNICZE_TABELE/najwyzsza_temp_4msc.csv', index=False)
najnizsza_temp.to_csv('WYNIKI/POMOCNICZE_TABELE/najnizsza_temp_4msc.csv', index=False)
najwieksze_opady.to_csv('WYNIKI/POMOCNICZE_TABELE/najwieksze_opady_4msc.csv', index=False)
najmniejsze_opady.to_csv('WYNIKI/POMOCNICZE_TABELE/najmniejsze_opady_4msc.csv', index=False)
najsilniejszy_wiatr.to_csv('WYNIKI/POMOCNICZE_TABELE/najsilniejszy_wiatr_4msc.csv', index=False)
najslabszy_wiatr.to_csv('WYNIKI/POMOCNICZE_TABELE/najslabszy_wiatr_4msc.csv', index=False)

starts = rowery[['start_station_name', 'start_lat', 'start_lng']].dropna().rename(columns={'start_station_name': 'nazwa_stacji', 'start_lat': 'szerokosc_geo', 'start_lng': 'dlugosc_geo'})
ends = rowery[['end_station_name', 'end_lat', 'end_lng']].dropna().rename(columns={'end_station_name': 'nazwa_stacji', 'end_lat': 'szerokosc_geo', 'end_lng': 'dlugosc_geo'})
stacje = pd.concat([starts, ends]).drop_duplicates(subset=['nazwa_stacji'])

def stworz_mape(trasy, nazwa_pliku, opis_pogody, kolumna_pogody, jednostka):
    mapa = folium.Map(location=[40.720, -74.040], zoom_start=14)
    min_val, max_val = trasy['liczba_tras'].min(), trasy['liczba_tras'].max()
    if min_val == max_val: max_val += 1
    skala_kolorow = cm.LinearColormap(colors=['#ff9999', '#ff0000', '#8b0000'], vmin=min_val, vmax=max_val, caption=f'Liczba przejazdów ({opis_pogody})')
    skala_kolorow.add_to(mapa)
    uzywane_stacje = set(trasy['start_station_name']).union(set(trasy['end_station_name']))
    for nazwa in uzywane_stacje:
        stacja = stacje[stacje['nazwa_stacji'] == nazwa]
        if not stacja.empty:
            folium.Marker([stacja.iloc[0]['szerokosc_geo'], stacja.iloc[0]['dlugosc_geo']], popup=nazwa, icon=folium.Icon(color='red' if any(s in opis_pogody for s in ['Wysok', 'Siln']) else 'blue', icon='bicycle', prefix='fa')).add_to(mapa)
    for _, row in trasy.iterrows():
        start = stacje[stacje['nazwa_stacji'] == row['start_station_name']]
        koniec = stacje[stacje['nazwa_stacji'] == row['end_station_name']]
        if not start.empty and not koniec.empty:
            folium.PolyLine([[start.iloc[0]['szerokosc_geo'], start.iloc[0]['dlugosc_geo']], [koniec.iloc[0]['szerokosc_geo'], koniec.iloc[0]['dlugosc_geo']]], color=skala_kolorow(row['liczba_tras']), weight=3, opacity=0.8, tooltip=f"{row['start_station_name']} -> {row['end_station_name']} ({row['liczba_tras']} przejazdów, {kolumna_pogody}: {row[kolumna_pogody]:.2f} {jednostka})").add_to(mapa)
    mapa.save(f'WYNIKI/MAPY/pogoda/{nazwa_pliku}')

def stworz_wykres_top_10_pogoda(trasy, tytul, sciezka):
    top = trasy.sort_values(by='liczba_tras', ascending=False).head(10).copy()
    top['trasa'] = top['start_station_name'] + " - " + top['end_station_name']
    plt.figure(figsize=(12, 6))
    plt.barh(top['trasa'], top['liczba_tras'], color='salmon')
    plt.xlabel('Liczba przejazdów')
    plt.title(tytul)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(sciezka)
    plt.close()

pary = [
    (najwyzsza_temp, 'mapa_najwyzsza_temp_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_temp_wysoka.png', 'Wysoka temperatura', 'temperatura', '°C'),
    (najnizsza_temp, 'mapa_najnizsza_temp_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_temp_niska.png', 'Niska temperatura', 'temperatura', '°C'),
    (najwieksze_opady, 'mapa_najwieksze_opady_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_opady_duze.png', 'Wysokie opady', 'opady', 'mm'),
    (najmniejsze_opady, 'mapa_najmniejsze_opady_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_opady_male.png', 'Małe opady', 'opady', 'mm'),
    (najsilniejszy_wiatr, 'mapa_najsilniejszy_wiatr_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_wiatr_silny.png', 'Silny wiatr', 'wiatr', 'km/h'),
    (najslabszy_wiatr, 'mapa_najslabszy_wiatr_4msc.html', 'WYNIKI/WYKRESY/pogoda/wykres_wiatr_slaby.png', 'Słaby wiatr', 'wiatr', 'km/h')
]

for df, m_plik, w_plik, opis, kol, jedn in pary:
    stworz_mape(df, m_plik, opis, kol, jedn)
    stworz_wykres_top_10_pogoda(df, f"Top 10 tras - {opis}", w_plik)